# Pathology-only topics and continuous patient risk

Standalone exploratory analysis of the platinum-trained, transferred XGBoost score at ADT +0 / +90 / +180 days.
Run manuscript notebook 04 first. This notebook reads its `transfer_scores.csv`, the pathology parquet, and optional
precomputed Clinical ModernBERT embeddings directly; it does **not** read Elio's extreme-decile manifest or create
patient exports. All finite scored patients are eligible, including the middle of the risk distribution.

The [Elio data request / high vs low risk patient extraction](../patient_extraction/01_high_vs_low_risk_transfer_pxs.ipynb)
is separate. Aggregate analysis outputs go to `<COMPASS data root>/testing/pathology_topics_vs_risk/`.

The existing term/lexicon/topic contrasts and text classifiers are retained for comparison. The **continuous topic
association** section uses the original numeric `risk_score`, without binning: note-level binomial GEE with patient
clusters, and patient-level topic fractions versus risk by Spearman correlation with patient permutations.
Topic discovery never receives risk scores. Rank percentiles remain only in the legacy term/lexicon sensitivity tests.

Outputs: `pathology_term_enrichment.csv`, `pathology_topics.csv`, `pathology_topic_enrichment.csv`,
`pathology_topic_continuous_association.csv`, `pathology_topic_risk_curves.csv`, `pathology_topic_continuous_coverage.csv`,
`pathology_lexicon_enrichment.csv`, `pathology_score_association.csv`, and `pathology_text_classifier.csv`.
BERTopic and its continuous analyses skip with an explanation if embeddings or optional packages are unavailable.
Only aggregate tables/curves are saved; no patient IDs, raw report text, representative documents, or fitted topic model.

**Timing:** all pathology dates (including undated and post-landmark reports) are included, matching the original analysis.
These are descriptive associations with risk, not prospective validation or causal effects. Later pathology may reflect
progression/treatment. A prospective analysis would need reports restricted to those available by each landmark.

For pooled and per-note-type analyses of Clinician, Imaging, and Pathology notes, use
[14_clinical_notes_topics_vs_risk.ipynb](14_clinical_notes_topics_vs_risk.ipynb).


## Configuration

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common import federated_scoring as fs

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
PATHOLOGY_PATH = Path(os.environ.get(
    "PROFILE_PATHOLOGY_NOTES",
    "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/CLINICAL_NOTES/PATHOLOGY_NOTES.parquet"))
OUTPUT_DIR = DATA_ROOT / "testing" / "pathology_topics_vs_risk"

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # the model is platinum-trained; risk_score is identical across endpoints
LANDMARKS = (0, 90, 180)
DECILE = 0.10                      # top and bottom 10% at each landmark

# Pathology term enrichment (top / bottom decile vs middle 80%)
TERM_MIN_PATIENTS = 5              # a term must appear in >= this many patients at the landmark
TERM_NGRAMS = (1, 2)               # unigrams and bigrams
TERM_FDR = 0.05                    # Benjamini-Hochberg threshold for the displayed tables
TERM_TOP_N = 25                    # terms shown per landmark x group

# Stratified sensitivity tests (Mantel-Haenszel, van Elteren): report-count band x any non-prostate specimen
ADJUST_REPORT_BANDS = (1, 2, 4)    # band lower edges: 1, 2-3, 4+ reports
SPECIMEN_HEADER_CHARS = 300        # report characters (after PROC_DESC) searched for the specimen site

# Cross-validated text classifiers vs a permutation null
TEXT_C = 1.0                       # inverse regularization strength
TEXT_FOLDS = 5
TEXT_PERMUTATIONS = 100
TEXT_SEED = 42
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))

# Optional BERTopic on the same reports, from precomputed Clinical ModernBERT embeddings (read-only, from the
# clinical_text_embedding_project's knit_embeddings.py). The section skips if packages or files are missing.
CTEP_PROCESSED_DIR = Path(os.environ.get(
    "CTEP_DATA_PATH", "/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/"
)) / "batched_datasets" / "processed_datasets"
EMBED_METADATA_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_metadata.parquet"
EMBED_ARRAY_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_as_array.npy.zst"
TOPIC_MIN_SIZE = 30                # HDBSCAN min_cluster_size, in reports
TOPIC_SEED = 42                    # UMAP random_state

# Continuous topic association: raw score, standardized across eligible patients within each landmark.
CONTINUOUS_MIN_PATIENTS = 30       # minimum independent patient clusters for asymptotic GEE inference
CONTINUOUS_MIN_TOPIC_PATIENTS = 5 # require patients with and without the topic
TOPIC_PERMUTATIONS = 1999         # patient-level Spearman independence null; increase for finer p-values


## Risk scores (all patients)

One full-model XGBoost score per patient and landmark. Invalid scores are excluded. Decile labels support the legacy
contrasts only; coincident cutoffs leave all finite scores in the middle group and do not block continuous analyses.


In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
print(scores.groupby(["model_family", "config"]).size())

sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"].isin(LANDMARKS))]
sel = sel[[fs.ID_COL, "landmark_days", "risk_score"]]
dup = sel.duplicated([fs.ID_COL, "landmark_days"], keep=False)
assert not dup.any(), f"{dup.sum()} rows: more than one {MODEL_FAMILY} model/arm per patient and landmark; narrow the filter above"
assert set(sel["landmark_days"]) == set(LANDMARKS), f"landmarks found: {sorted(sel['landmark_days'].unique())}"

def decile_flags(g):
    if not 0 < DECILE < 0.5:
        raise ValueError("DECILE must be between 0 and 0.5 (exclusive).")
    g = g.copy()
    risk = pd.to_numeric(g["risk_score"], errors="coerce").astype(float)
    risk = risk.where(np.isfinite(risk))
    valid = risk.notna()
    landmark = g["landmark_days"].iloc[0]
    lo, hi = risk[valid].quantile([DECILE, 1 - DECILE]) if valid.any() else (np.nan, np.nan)
    if not valid.any() or lo >= hi:
        print(f"Landmark {landmark}: no distinct decile cutoffs; legacy extreme-group contrasts unavailable.")
    g["risk_score"] = risk
    g["risk_percentile"] = risk.rank(pct=True, method="average")
    # Only the two extreme deciles are labelled; everything in between stays blank.
    g["decile_group"] = pd.Series(pd.NA, index=g.index, dtype="string")
    if lo < hi:
        g.loc[risk >= hi, "decile_group"] = "top"
        g.loc[risk <= lo, "decile_group"] = "bottom"
    g.loc[~valid, "decile_group"] = "not scored"
    g.attrs.update(lo=lo, hi=hi)
    return g

by_lm = {lm: decile_flags(g) for lm, g in sel.groupby("landmark_days")}
cutoffs = pd.DataFrame({lm: {"n_scored": int(g["risk_score"].notna().sum()),
                            "n_not_scored": int(g["risk_score"].isna().sum()),
                            "bottom_cutoff": g.attrs["lo"], "top_cutoff": g.attrs["hi"],
                            "n_top": int((g["decile_group"] == "top").sum()),
                            "n_bottom": int((g["decile_group"] == "bottom").sum())} for lm, g in by_lm.items()}).T
cutoffs

## Pathology term enrichment: top / bottom decile vs middle 80%, and top vs bottom

At each landmark, patients fall into three groups by their score at that landmark: **top** decile, **bottom** decile, and
**middle** (scored, in neither decile; ties at a cutoff stay in the extreme groups, so this is at most 80%). Four
contrasts per landmark, each reading "overrepresented in `group` relative to `control`": top vs middle, bottom vs middle,
top vs bottom and bottom vs top. Top vs bottom is run in both directions because the test is one-sided. Each patient's
group can differ between landmarks.

- **Reports:** every pathology report of each scored patient, whatever its date (before or after ADT start and the
  landmark, or undated). Groups still come from each landmark's score, so the landmarks differ only in who is scored
  and which decile they fall in. Later reports include post-progression pathology, so terms can reflect the outcome
  the model predicts rather than what was on file when the patient was scored. Patients with no pathology report drop
  out of that landmark's denominators.
- **Terms:** lowercased letter-only unigrams and bigrams (numbers, dates and IDs drop out), English stop words removed,
  counted within each report so bigrams never span two reports. A term is kept if it appears in at least
  `TERM_MIN_PATIENTS` patients at the landmark, which reduces rare strings but does not guarantee de-identification; keep outputs protected.
- **Test:** a patient "has" a term if any of their reports contains it. For each contrast, one-sided Fisher's
  exact test for overrepresentation in `group`, Benjamini-Hochberg q-values within each contrast, and the odds ratio
  with a +0.5 correction. Mean patient-level TF-IDF (sublinear term frequency over each patient's pooled counts) is reported
  for each group as a descriptive column. The `*_mh` columns repeat the test stratified by report count and specimen
  site (Mantel-Haenszel; see below).

In [ ]:
import re

import polars as pl

TERM_GROUPS = ("top", "bottom")
CONTROL_GROUP = "middle"
# (group, control): one-sided tests for terms / topics overrepresented in group relative to control.
CONTRASTS = (("top", CONTROL_GROUP), ("bottom", CONTROL_GROUP), ("top", "bottom"), ("bottom", "top"))

# --- Specimen site (for the stratified sensitivity test): the site named first in PROC_DESC + the report's opening.
SPECIMEN_SITES = (
    ("prostate", re.compile(r"\b(?:prostat\w*|turp|transurethral|seminal\s+vesicles?)\b")),
    ("non-prostate", re.compile(r"\b(?:bone|marrow|liver|hepatic|lymph|nodes?|nodal|lung|pulmonary|pleura\w*|adrenal"
                                r"|brain|dura\w*|spine|spinal|vertebra\w*|soft\s+tissue|peritone\w*|omentum|omental)\b")),
)


def specimen_site(header):
    """'prostate', 'non-prostate' or 'unspecified': whichever site is named first in a lowercased header."""
    hits = [(m.start(), site) for site, rx in SPECIMEN_SITES for m in [rx.search(header)] if m]
    return min(hits)[1] if hits else "unspecified"


# Per-landmark group for every scored patient; a blank decile group means scored but in neither decile.
groups_by_lm = {}
for lm, g in by_lm.items():
    ids = pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64"), name="DFCI_MRN")
    grp = pd.Series(g["decile_group"].fillna(CONTROL_GROUP).to_numpy(), index=ids, name="group")
    groups_by_lm[lm] = grp[grp.ne("not scored")]

scored_ids = sorted({int(i) for grp in groups_by_lm.values() for i in grp.index.dropna()})
# RPT_ID (when present) links reports to the optional embeddings; PROC_DESC (when present) helps type the specimen.
note_cols = ["DFCI_MRN", "EVENT_DATE", "RPT_TEXT"]
schema = pl.scan_parquet(PATHOLOGY_PATH).collect_schema().names()
note_cols += [c for c in ("RPT_ID", "PROC_DESC") if c in schema]
all_notes = (pl.scan_parquet(PATHOLOGY_PATH)
             .select(note_cols)
             .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(scored_ids))
             .collect().to_pandas())
all_notes["DFCI_MRN"] = pd.to_numeric(all_notes["DFCI_MRN"], errors="coerce").astype("Int64")
all_notes["RPT_TEXT"] = all_notes["RPT_TEXT"].fillna("").astype(str)
all_notes["note_row"] = np.arange(len(all_notes))      # row position, used to look up per-report lexicon hits

# Specimen site, once per report.
header = all_notes["RPT_TEXT"].str[:SPECIMEN_HEADER_CHARS]
if "PROC_DESC" in all_notes:
    header = all_notes["PROC_DESC"].fillna("").astype(str) + " " + header
all_notes["specimen_site"] = header.str.lower().map(specimen_site) if len(all_notes) else pd.Series(dtype=object)

# Every report of the patients scored at each landmark, whatever its date.
# Each patient's stratum: report count band (ADJUST_REPORT_BANDS) x any non-prostate-site report.
reports_by_lm, strata_by_lm, size_rows = {}, {}, []
for lm, grp in groups_by_lm.items():
    rep_lm = all_notes.loc[all_notes["DFCI_MRN"].isin(grp.index), ["DFCI_MRN", "RPT_TEXT", "specimen_site", "note_row"]
                           + note_cols[3:]].reset_index(drop=True)
    reports_by_lm[lm] = rep_lm
    per = (rep_lm.assign(non_prostate=rep_lm["specimen_site"].eq("non-prostate"))
           .groupby("DFCI_MRN").agg(reports=("non_prostate", "size"), non_prostate=("non_prostate", "any")))
    band = np.searchsorted(ADJUST_REPORT_BANDS, per["reports"].to_numpy(), side="right") - 1
    strata_by_lm[lm] = pd.Series(band * 2 + per["non_prostate"].to_numpy(dtype=int), index=per.index, name="stratum")
    has_report = grp.index.isin(rep_lm["DFCI_MRN"])
    for name in (*TERM_GROUPS, CONTROL_GROUP):
        in_group = grp.eq(name).to_numpy()
        members = per.index.isin(grp.index[in_group])
        size_rows.append({"landmark": lm, "group": name, "patients_scored": int(in_group.sum()),
                          "patients_with_report": int((in_group & has_report).sum()),
                          "reports": int(rep_lm["DFCI_MRN"].isin(grp.index[in_group]).sum()),
                          "median_reports": float(per.loc[members, "reports"].median()) if members.any() else np.nan,
                          "patients_with_non_prostate_report": int(per.loc[members, "non_prostate"].sum())})

sites = all_notes["specimen_site"].value_counts()
print(f"Specimen site of the {len(all_notes):,} pathology reports of scored patients (all dates): "
      + ", ".join(f"{site} {int(sites.get(site, 0)):,}" for site in ("prostate", "non-prostate", "unspecified")))
group_sizes = pd.DataFrame(size_rows).set_index(["landmark", "group"])
group_sizes


In [ ]:
from scipy import sparse
from scipy.stats import hypergeom, norm
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# Applied after lowercasing: letters only, >= 3 characters.
TOKEN_PATTERN = r"(?u)\b[a-z][a-z]{2,}\b"
MH_COLS = ["odds_ratio_mh", "p_value_mh", "q_value_mh"]
ENRICHMENT_COLS = ["landmark", "group", "control", "term", "n_group", "n_control", "group_with_term", "control_with_term",
                   "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS,
                   "mean_tfidf_group", "mean_tfidf_control"]


def bh_qvalues(p):
    """Benjamini-Hochberg q-values over the finite p values; NaN stays NaN (cluster scipy predates
    false_discovery_control)."""
    p = np.asarray(p, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if ok.size:
        order = ok[np.argsort(p[ok])]
        ranked = p[order] * ok.size / np.arange(1, ok.size + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def patient_term_matrices(reports, ids):
    """Patient x term presence and TF-IDF, with terms counted within each report and summed per patient.
    """
    text = reports["RPT_TEXT"].to_numpy()
    vec = CountVectorizer(lowercase=True, stop_words="english", token_pattern=TOKEN_PATTERN,
                          ngram_range=TERM_NGRAMS, min_df=TERM_MIN_PATIENTS)
    try:
        # Report-level min_df is a safe prefilter: a term in >= k patients is in >= k reports.
        counts = vec.fit_transform(text)
    except ValueError:  # no term survives (or fewer reports than TERM_MIN_PATIENTS)
        return np.array([], dtype=object), None, None
    rows = ids.get_indexer(reports["DFCI_MRN"])
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))),
                                   shape=(len(ids), len(rows)))
    patient_counts = (to_patient @ counts).tocsc()
    present = (patient_counts > 0).astype(np.int64)
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return np.array([], dtype=object), None, None
    patient_counts, present = patient_counts[:, keep], present[:, keep]
    tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(patient_counts)
    return vec.get_feature_names_out()[keep], present.tocsr(), tfidf.tocsr()


def colsum(m):
    return np.asarray(m.sum(axis=0)).ravel().astype(float)


def mantel_haenszel(present, in_group, control, strata):
    """One-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio per feature column, across strata.

    A stratum without patients from both sides carries no information and is skipped; if none is left, both are NaN.
    The odds ratio adds 0.5 to every cell of a stratum that has an empty cell.
    """
    n_feat = present.shape[1]
    A, E, V, R, S = (np.zeros(n_feat) for _ in range(5))
    informative = False
    for k in pd.unique(strata[in_group | control]):
        g, c = in_group & (strata == k), control & (strata == k)
        n1, n0 = int(g.sum()), int(c.sum())
        if not (n1 and n0):
            continue
        informative = True
        n = n1 + n0
        a, cc = colsum(present[g]), colsum(present[c])
        b, d, m = n1 - a, n0 - cc, a + cc
        A += a
        E += n1 * m / n
        V += n1 * n0 * m * (n - m) / (n ** 2 * (n - 1))
        h = 0.5 * ((a == 0) | (b == 0) | (cc == 0) | (d == 0))
        R += (a + h) * (d + h) / (n + 4 * h)
        S += (b + h) * (cc + h) / (n + 4 * h)
    if not informative:
        return np.full(n_feat, np.nan), np.full(n_feat, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(V > 0, norm.sf((A - E - 0.5) / np.sqrt(V)), 1.0)
    return R / S, p


def fisher_enrichment(present, label, features, landmark, strata=None):
    """One-sided Fisher overrepresentation of each feature column for each (group, control) in CONTRASTS.

    present: patients x features 0/1 sparse matrix; label: each patient's group; strata (optional): each patient's
    stratum for the Mantel-Haenszel columns (NaN without it). Rows come out in CONTRASTS order, len(features) rows
    per contrast whose group and control both have patients.
    """
    out = []
    for name, ref in CONTRASTS:
        in_group, control = label == name, label == ref
        n1, n0 = int(in_group.sum()), int(control.sum())
        if n1 == 0 or n0 == 0:
            continue
        a, c = colsum(present[in_group]), colsum(present[control])
        p = hypergeom.sf(a - 1, n1 + n0, a + c, n1)       # P(X >= a): one-sided Fisher
        if strata is None:
            or_mh = p_mh = np.full(len(features), np.nan)
        else:
            or_mh, p_mh = mantel_haenszel(present, in_group, control, np.asarray(strata))
        out.append(pd.DataFrame({
            "landmark": landmark, "group": name, "control": ref, "feature": features, "n_group": n1, "n_control": n0,
            "group_with_feature": a.astype(int), "control_with_feature": c.astype(int),
            "pct_group": a / n1, "pct_control": c / n0,
            "odds_ratio": ((a + 0.5) * (n0 - c + 0.5)) / ((n1 - a + 0.5) * (c + 0.5)),
            "p_value": p, "q_value": bh_qvalues(p),
            "odds_ratio_mh": or_mh, "p_value_mh": p_mh, "q_value_mh": bh_qvalues(p_mh),
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def sort_contrasts(df):
    """Landmark, then CONTRASTS order, then p value and odds ratio."""
    rank = {pair: i for i, pair in enumerate(CONTRASTS)}
    order = pd.Series([rank[pair] for pair in zip(df["group"], df["control"])], index=df.index, dtype=float)
    return (df.assign(_contrast=order)
            .sort_values(["landmark", "_contrast", "p_value", "odds_ratio"], ascending=[True, True, True, False])
            .drop(columns="_contrast").reset_index(drop=True))


def contrast_title(lm, name, ref):
    return f"Landmark {lm} · {name} decile vs {'middle' if ref == CONTROL_GROUP else ref + ' decile'}"


term_matrices = {}   # landmark -> (patient ids, terms, presence, TF-IDF); reused by the score association cell


def term_enrichment(reports, groups, landmark):
    """Fisher overrepresentation of each term for each contrast in CONTRASTS at one landmark."""
    ids = pd.Index(sorted(reports["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 report
    terms, present, tfidf = patient_term_matrices(reports, ids)
    term_matrices[landmark] = (ids, terms, present, tfidf)
    if not len(terms):
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    label = groups.reindex(ids).to_numpy()
    res = fisher_enrichment(present, label, terms, landmark, strata_by_lm[landmark].reindex(ids).to_numpy())
    if res.empty:
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    mean_tfidf = {name: np.asarray(tfidf[label == name].mean(axis=0)).ravel()
                  for name in set(res["group"]) | set(res["control"])}
    pairs = list(res[["group", "control"]].drop_duplicates().itertuples(index=False))
    res["mean_tfidf_group"] = np.concatenate([mean_tfidf[g] for g, _ in pairs])
    res["mean_tfidf_control"] = np.concatenate([mean_tfidf[c] for _, c in pairs])
    return res.rename(columns={"feature": "term", "group_with_feature": "group_with_term",
                               "control_with_feature": "control_with_term"})


enrichment = pd.concat(
    [term_enrichment(reports_by_lm[lm], groups_by_lm[lm], lm) for lm in LANDMARKS if lm in reports_by_lm],
    ignore_index=True,
).reindex(columns=ENRICHMENT_COLS)
enrichment = sort_contrasts(enrichment)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
enrichment_path = OUTPUT_DIR / "pathology_term_enrichment.csv"
enrichment.to_csv(enrichment_path, index=False)
print(f"wrote {enrichment_path} ({len(enrichment):,} term x contrast rows)")


In [ ]:
try:
    from IPython.display import display
except ImportError:  # plain-Python runs
    display = print

show_cols = ["term", "group_with_term", "n_group", "pct_group", "control_with_term", "n_control", "pct_control",
             "odds_ratio", "q_value", "odds_ratio_mh", "q_value_mh", "mean_tfidf_group", "mean_tfidf_control"]
for lm in LANDMARKS:
    for name, ref in CONTRASTS:
        sub = enrichment[(enrichment["landmark"] == lm) & (enrichment["group"] == name) & (enrichment["control"] == ref)]
        title = contrast_title(lm, name, ref)
        if sub.empty:
            print(f"\n{title}: no terms (too few patients or reports)")
            continue
        sig = sub[sub["q_value"] < TERM_FDR]
        note = f"q < {TERM_FDR}: {len(sig):,} terms"
        if sig.empty:
            sig, note = sub, f"none at q < {TERM_FDR}; top {TERM_TOP_N} by p shown"
        top = sig.sort_values(["p_value", "odds_ratio"], ascending=[True, False]).head(TERM_TOP_N)
        print(f"\n{title} ({note})")
        display(top[show_cols].round(3).reset_index(drop=True))

## Optional: BERTopic topics in the same reports

Groups and reports (all dates) are the same as above. Instead of single terms, reports are clustered into topics,
and topics are tested for overrepresentation in the same four contrasts.

- **Embeddings:** precomputed, mean-pooled Clinical ModernBERT embeddings of each pathology report, read (never written)
  from the clinical_text_embedding_project (`full_clinical_notes_embeddings_as_array.npy.zst` and its metadata parquet),
  joined on `RPT_ID`. No language model is loaded and nothing is downloaded. Reports outside that project's cohort have
  no embedding; the coverage table shows how many patients each contrast keeps.
- **Model:** one BERTopic fit on the union of embedded reports across landmarks (each report once), so topic
  numbers mean the same thing at every landmark. UMAP (5 dimensions, cosine, `TOPIC_SEED`) then HDBSCAN
  (`TOPIC_MIN_SIZE` reports). Topic words use only the terms that appear in at least `TERM_MIN_PATIENTS` patients.
- **Test:** a patient "has" a topic if any of their embedded reports is assigned to it. Outlier reports
  (topic -1) are not a feature, but their patients stay in the denominators. Denominators are patients with at least one
  embedded report. Topics in fewer than `TERM_MIN_PATIENTS` patients at a landmark are dropped there.
  Same one-sided Fisher test, BH q-values and odds ratio as the terms.
- **PHI:** BERTopic keeps representative documents (raw report text) inside the fitted model. They are dropped from
  every table and the model is never saved; only topic sizes and top words are written.

The section needs `bertopic`, `umap-learn`, `hdbscan` and `zstandard` (commented out in `requirements.txt`). Without
them, or without the embedding files, it prints why it skipped and writes nothing; the term enrichment above is unaffected.
Loading the embeddings needs only `zstandard`; they are reused by the embedding classifier below even when BERTopic is
not installed.

In [ ]:
import importlib
import io

TOPIC_PACKAGES = {"zstandard": "zstandard", "bertopic": "bertopic", "umap": "umap-learn", "hdbscan": "hdbscan"}


def optional_import(name):
    try:
        return importlib.import_module(name)
    except Exception:   # ImportError, or a broken install (e.g. numba) that fails on import
        return None


def rpt_key(values):
    """RPT_ID as a string join key; whole-number IDs lose any float formatting ("12.0" -> "12")."""
    s = pd.Series(values).reset_index(drop=True)
    num = pd.to_numeric(s, errors="coerce")
    whole = num.notna() & np.isfinite(num) & (num % 1 == 0)
    key = s.astype("string").str.strip()
    key[whole] = num[whole].astype("int64").astype(str)
    return key.where(s.notna())


def _read_exact(fh, n):
    parts, got = [], 0
    while got < n:
        chunk = fh.read(n - got)
        if not chunk:
            raise ValueError("embedding array ended before its header's shape")
        parts.append(chunk)
        got += len(chunk)
    return b"".join(parts)


def read_embedding_rows(path, rows, zstd, chunk_rows=16384):
    """Rows `rows` (sorted, unique) of a zstd-compressed 2-D .npy array, as float32, in one streaming pass.

    The full array (every note type for the embedding cohort) is never held in memory: it is decompressed in
    chunk_rows blocks and only the requested rows are kept.
    """
    rows = np.asarray(rows, dtype=np.int64)
    with open(path, "rb") as fh, zstd.ZstdDecompressor().stream_reader(fh) as raw:
        version = np.lib.format.read_magic(raw)
        readers = {(1, 0): np.lib.format.read_array_header_1_0, (2, 0): np.lib.format.read_array_header_2_0}
        if version not in readers:
            raise ValueError(f"unsupported .npy version {version}")
        shape, fortran_order, dtype = readers[version](raw)
        if fortran_order or len(shape) != 2 or dtype.hasobject:
            raise ValueError(f"expected a C-order 2-D numeric array, got shape {shape}")
        n_rows, dim = shape
        if rows.size and (rows[0] < 0 or rows[-1] >= n_rows):
            raise ValueError(f"EMBEDDING_INDEX outside the array's {n_rows:,} rows")
        out = np.empty((rows.size, dim), dtype=np.float32)
        start, stop = 0, int(rows[-1]) + 1 if rows.size else 0
        while start < stop:
            k = min(chunk_rows, stop - start)
            block = np.frombuffer(_read_exact(raw, k * dim * dtype.itemsize), dtype=dtype).reshape(k, dim)
            lo, hi = np.searchsorted(rows, [start, start + k])
            out[lo:hi] = block[rows[lo:hi] - start]
            start += k
    return out


topic_mods = {name: optional_import(name) for name in TOPIC_PACKAGES}
missing = [pip for name, pip in TOPIC_PACKAGES.items() if topic_mods[name] is None]

# Loading the embeddings needs only zstandard; the text classifier at the end of the notebook reuses them.
embed_skip = None
if topic_mods["zstandard"] is None:
    embed_skip = "packages not installed: zstandard"
elif not (EMBED_METADATA_PATH.exists() and EMBED_ARRAY_PATH.exists()):
    embed_skip = f"precomputed embeddings not found under {EMBED_ARRAY_PATH.parent}"
elif "RPT_ID" not in all_notes:
    embed_skip = "pathology notes have no RPT_ID to join the embeddings on"

embed_docs = report_embeddings = None   # embed_docs row i <-> report_embeddings row i
if embed_skip is None:
    # Each report once, across landmarks (the landmarks only differ in which patients are scored).
    embed_docs = (pd.concat(reports_by_lm.values(), ignore_index=True)
                  .assign(rpt_key=lambda d: rpt_key(d["RPT_ID"]).to_numpy())
                  .dropna(subset=["rpt_key"]).drop_duplicates("rpt_key", ignore_index=True))
    meta = (pl.scan_parquet(EMBED_METADATA_PATH)
            .filter(pl.col("NOTE_TYPE") == "Pathology")
            .select("RPT_ID", "EMBEDDING_INDEX")
            .collect().to_pandas())
    meta["rpt_key"] = rpt_key(meta["RPT_ID"]).to_numpy()
    meta = meta.dropna(subset=["rpt_key", "EMBEDDING_INDEX"]).drop_duplicates("rpt_key")
    n_window = len(embed_docs)
    embed_docs = (embed_docs.merge(meta[["rpt_key", "EMBEDDING_INDEX"]], on="rpt_key", how="inner",
                                   validate="one_to_one")
                  .sort_values("EMBEDDING_INDEX", ignore_index=True))
    print(f"{len(embed_docs):,} of {n_window:,} pathology reports have a precomputed embedding")
    if embed_docs.empty:
        embed_skip, embed_docs = "no pathology report has a precomputed embedding", None
    else:
        report_embeddings = read_embedding_rows(EMBED_ARRAY_PATH, embed_docs["EMBEDDING_INDEX"].astype("int64").to_numpy(),
                                                topic_mods["zstandard"])

topic_skip = None
if missing:
    topic_skip = f"packages not installed: {', '.join(missing)}"
elif embed_skip is not None:
    topic_skip = embed_skip
elif len(embed_docs) <= 2 * TOPIC_MIN_SIZE:
    topic_skip = f"only {len(embed_docs):,} embedded reports (need more than {2 * TOPIC_MIN_SIZE})"
else:
    # Topic words come only from terms in >= TERM_MIN_PATIENTS patients, the same rule as the term enrichment.
    vocab, _, _ = patient_term_matrices(embed_docs, pd.Index(sorted(embed_docs["DFCI_MRN"].unique()), dtype="Int64"))
    if not len(vocab):
        topic_skip = f"no term appears in >= {TERM_MIN_PATIENTS} patients' embedded reports"

topic_docs = None
if topic_skip is None:
    topic_model = topic_mods["bertopic"].BERTopic(
        embedding_model=None,   # embeddings are passed in: no backend is selected or downloaded
        umap_model=topic_mods["umap"].UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                                           random_state=TOPIC_SEED),
        hdbscan_model=topic_mods["hdbscan"].HDBSCAN(min_cluster_size=TOPIC_MIN_SIZE, metric="euclidean",
                                                    cluster_selection_method="eom", prediction_data=False),
        vectorizer_model=CountVectorizer(vocabulary=list(vocab), ngram_range=TERM_NGRAMS, lowercase=True,
                                         token_pattern=TOKEN_PATTERN),
        calculate_probabilities=False,
        verbose=False,
    )
    assigned, _ = topic_model.fit_transform(embed_docs["RPT_TEXT"].tolist(), embeddings=report_embeddings)
    topic_docs = embed_docs.assign(topic=np.asarray(assigned, dtype=int))
    info = topic_model.get_topic_info()
    words = {int(t): " ".join(w for w, _ in (topic_model.get_topic(t) or [])) for t in info["Topic"]}
    del topic_model, info   # the model holds representative report text: never saved or shown
    topic_table = (topic_docs.groupby("topic")
                   .agg(n_reports=("rpt_key", "size"), n_patients=("DFCI_MRN", "nunique"))
                   .reset_index())
    topic_table["top_words"] = topic_table["topic"].map(words).fillna("")
    topic_table.loc[topic_table["topic"].eq(-1), "top_words"] = "(outliers: no topic)"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    topics_path = OUTPUT_DIR / "pathology_topics.csv"
    topic_table.to_csv(topics_path, index=False)
    print(f"wrote {topics_path} ({topic_table['topic'].ge(0).sum():,} topics, "
          f"{topic_table.loc[topic_table['topic'].eq(-1), 'n_reports'].sum():,} outlier reports)")
else:
    print(f"BERTopic section skipped ({topic_skip}); no topic files written.")


In [ ]:
TOPIC_COLS = ["landmark", "group", "control", "topic", "top_words", "n_group", "n_control", "group_with_topic",
              "control_with_topic", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS]


def topic_enrichment(docs, groups, landmark):
    """Fisher overrepresentation of each topic for each contrast in CONTRASTS at one landmark."""
    ids = pd.Index(sorted(docs["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 embedded report
    topics = np.array(sorted(t for t in docs["topic"].unique() if t >= 0))
    hits = docs[docs["topic"].ge(0)]
    present = sparse.csr_matrix(
        (np.ones(len(hits)), (ids.get_indexer(hits["DFCI_MRN"]), np.searchsorted(topics, hits["topic"]))),
        shape=(len(ids), len(topics)))
    present = (present > 0).astype(np.int64).tocsc()
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = fisher_enrichment(present[:, keep].tocsr(), groups.reindex(ids).to_numpy(), topics[keep], landmark,
                            strata_by_lm[landmark].reindex(ids).to_numpy())
    if res.empty:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = res.rename(columns={"feature": "topic", "group_with_feature": "group_with_topic",
                              "control_with_feature": "control_with_topic"})
    return res.assign(top_words=res["topic"].map(topic_table.set_index("topic")["top_words"]))


if topic_docs is None:
    print("BERTopic section skipped; see the previous cell.")
else:
    topic_frames, coverage_rows = [], []
    keys = topic_docs.set_index("rpt_key")
    for lm in LANDMARKS:
        if lm not in reports_by_lm:
            continue
        grp = groups_by_lm[lm]
        lm_keys = rpt_key(reports_by_lm[lm]["RPT_ID"]).dropna().drop_duplicates()
        docs_lm = keys.loc[keys.index.intersection(pd.Index(lm_keys)), ["DFCI_MRN", "topic"]].reset_index()
        topic_frames.append(topic_enrichment(docs_lm, grp, lm))
        for name in (*TERM_GROUPS, CONTROL_GROUP):
            members = grp.index[grp.eq(name).to_numpy()]
            in_group = docs_lm["DFCI_MRN"].isin(members)
            coverage_rows.append({"landmark": lm, "group": name,
                                  "patients_with_report": group_sizes.loc[(lm, name), "patients_with_report"],
                                  "patients_with_embedded_report": int(docs_lm.loc[in_group, "DFCI_MRN"].nunique()),
                                  "embedded_reports": int(in_group.sum()),
                                  "outlier_reports": int((in_group & docs_lm["topic"].eq(-1)).sum())})
    topic_enrich = sort_contrasts(pd.concat(topic_frames, ignore_index=True).reindex(columns=TOPIC_COLS))
    topic_enrich_path = OUTPUT_DIR / "pathology_topic_enrichment.csv"
    topic_enrich.to_csv(topic_enrich_path, index=False)
    print(f"wrote {topic_enrich_path} ({len(topic_enrich):,} topic x contrast rows)")

    topic_coverage = pd.DataFrame(coverage_rows).set_index(["landmark", "group"])
    display(topic_coverage)
    topic_show = ["topic", "top_words", "group_with_topic", "n_group", "pct_group", "control_with_topic",
                  "n_control", "pct_control", "odds_ratio", "q_value", "odds_ratio_mh", "q_value_mh"]
    for lm in LANDMARKS:
        for name, ref in CONTRASTS:
            sub = topic_enrich[(topic_enrich["landmark"] == lm) & (topic_enrich["group"] == name)
                               & (topic_enrich["control"] == ref)]
            title = f"{contrast_title(lm, name, ref)} (topics)"
            if sub.empty:
                print(f"\n{title}: no topics (too few patients or reports)")
                continue
            print(f"\n{title} (q < {TERM_FDR}: {int((sub['q_value'] < TERM_FDR).sum()):,} topics)")
            display(sub[topic_show].head(TERM_TOP_N).round(3).reset_index(drop=True))

## Continuous topic association: does topic frequency change with patient risk?

Yes: use every finite patient score and keep `risk_score` continuous. The models below fit **each topic vs all other
notes** separately at each landmark; topic -1 is included among other notes and in patient note-count denominators.
Only patients with an embedded pathology report are represented. Missing embeddings are not negative topic assignments;
the coverage table reports exclusions. The score is a model output, not a calibrated event probability.

- **Note-level binomial GEE (primary):** `I(note topic == k) ~ standardized raw risk score`, clustered by patient,
  with an independence working correlation and robust sandwich standard errors. The odds ratio is per one patient-level
  SD of risk, computed among eligible patients at that landmark. OR > 1 means the topic becomes more common in notes
  as risk increases. The adjusted sensitivity adds log(1 + embedded report count) and report specimen site (non-prostate
  and unspecified indicators). Constant covariates are omitted; collinear or unstable fits are marked as skipped.
  GEE handles correlated notes, but this estimates a **note-weighted** association: patients with more notes contribute more.
- **Patient-level sensitivity:** fraction of a patient's embedded notes assigned to each topic, correlated with the raw
  risk score using Spearman rho. Each patient contributes once. Two-sided permutation p-values shuffle whole patient scores
  under an independence null, preserving ties and each patient's full note collection. This is unadjusted and detects
  monotonic associations; it does not assume the score is linearly spaced. Set `TOPIC_PERMUTATIONS` higher for finer p-values.
- **Inference:** require `CONTINUOUS_MIN_PATIENTS` independent patient clusters and at least
  `CONTINUOUS_MIN_TOPIC_PATIENTS` patients contributing topic notes and other notes (these sets may overlap).
  Constant scores/fractions and failed fits are explicitly skipped. The patient minimum is a screening rule, not a guarantee
  of reliable asymptotic intervals. BH correction is across **all topics and landmarks within each method**; methods are
  separate analysis families. Skipped fits have no p/q-value. Topic discovery uncertainty is not included.
- **Curves:** unadjusted GEE fitted probabilities over the observed raw score range, with pointwise 95% robust intervals.
  There are no risk bins. These curves assume a linear effect on log odds and are not evidence of a nonlinear relationship.
  A spline of risk in the same clustered model is a useful later extension if the cohort supports the extra parameters.

All dates are still included, so associations can reflect later progression, sampling intensity, or treatment. Report-count
and specimen adjustment are descriptive sensitivities, not a causal correction. Rerun with notes available by the landmark
before interpreting topics as information available at prediction time. Scores are standardized separately per landmark;
ORs therefore refer to different score SDs and should not be treated as independent replication across landmarks.

Methods: [statsmodels GEE documentation](https://www.statsmodels.org/stable/gee.html),
[SciPy Spearman guidance](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html),
[paired permutation tests](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html).
Install `statsmodels>=0.14` for GEE; patient-level correlations still run without it.


In [ ]:
import warnings

from scipy.special import expit
from scipy.stats import permutation_test, rankdata

CONTINUOUS_COLS = [
    "landmark", "topic", "top_words", "model", "n_patients", "n_reports", "n_topic_reports",
    "n_patients_with_topic", "n_patients_with_other_reports", "risk_mean", "risk_sd",
    "beta_per_sd", "odds_ratio_per_sd", "ci95_low", "ci95_high", "spearman_rho",
    "p_value", "q_value", "status",
]
CURVE_COLS = ["landmark", "topic", "top_words", "risk_score", "probability", "ci95_low", "ci95_high"]


def continuous_topic_association(docs, risk, landmark, top_words):
    """One note per row; risk is a unique patient-indexed Series of raw scores.

    GEE targets note-level topic odds; Spearman targets equally weighted patient topic fractions.
    Outlier notes (-1) remain in denominators. Return aggregate effects and unadjusted fitted curves only.
    """
    if risk.index.has_duplicates:
        raise ValueError("Expected one risk score per patient at this landmark.")
    if TOPIC_PERMUTATIONS < 1:
        raise ValueError("TOPIC_PERMUTATIONS must be positive.")
    docs = docs.copy()
    docs["risk_score"] = pd.to_numeric(docs["DFCI_MRN"].map(risk), errors="coerce")
    docs = docs[np.isfinite(docs["risk_score"])].reset_index(drop=True)
    patients = docs.groupby("DFCI_MRN").agg(risk_score=("risk_score", "first"), n_reports=("topic", "size"))
    n_patients = len(patients)
    mean, sd = patients["risk_score"].mean(), patients["risk_score"].std(ddof=0)
    topics = sorted(t for t in docs["topic"].unique() if t >= 0)
    sm = optional_import("statsmodels.api")
    results, curves = [], []
    for topic in topics:
        y = docs["topic"].eq(topic).astype(float)
        counts = y.groupby(docs["DFCI_MRN"]).sum().reindex(patients.index, fill_value=0)
        fraction = counts / patients["n_reports"]
        base = dict(landmark=landmark, topic=topic, top_words=top_words.get(topic, ""),
                    n_patients=n_patients, n_reports=len(docs), n_topic_reports=int(y.sum()),
                    n_patients_with_topic=int(counts.gt(0).sum()),
                    n_patients_with_other_reports=int(counts.lt(patients["n_reports"]).sum()),
                    risk_mean=mean, risk_sd=sd)
        reason = None
        if n_patients < CONTINUOUS_MIN_PATIENTS:
            reason = "too few patient clusters"
        elif not np.isfinite(sd) or sd <= 0:
            reason = "constant risk score"
        elif min(base["n_patients_with_topic"], base["n_patients_with_other_reports"]) < CONTINUOUS_MIN_TOPIC_PATIENTS:
            reason = "too few patients with topic or other notes"
        for model in ("gee", "gee_adjusted", "patient_fraction_spearman"):
            row = dict(base, model=model, status="ok")
            if reason:
                row["status"] = "skipped: " + reason
            elif model == "patient_fraction_spearman":
                if fraction.nunique() < 2:
                    row["status"] = "skipped: constant patient topic fraction"
                else:
                    # Rank once; permute whole patients, never individual notes. Two-sided independence test.
                    x = rankdata(patients["risk_score"].to_numpy())
                    x = (x - x.mean()) / np.linalg.norm(x - x.mean())
                    f = rankdata(fraction.to_numpy())
                    f = (f - f.mean()) / np.linalg.norm(f - f.mean())
                    def statistic(permuted, axis=-1):
                        return np.sum(permuted * f, axis=axis)
                    test = permutation_test((x,), statistic, permutation_type="pairings", vectorized=True,
                                            n_resamples=TOPIC_PERMUTATIONS, batch=128, alternative="two-sided",
                                            random_state=TOPIC_SEED)
                    row.update(spearman_rho=float(test.statistic), p_value=float(test.pvalue))
            elif sm is None:
                row["status"] = "skipped: statsmodels not installed or unavailable"
            else:
                x = pd.DataFrame({"intercept": np.ones(len(docs)), "risk_z": (docs["risk_score"] - mean) / sd})
                if model == "gee_adjusted":
                    covariates = {
                        "log_report_count": np.log1p(docs["DFCI_MRN"].map(patients["n_reports"]).to_numpy()),
                        "non_prostate": docs["specimen_site"].eq("non-prostate").astype(float).to_numpy(),
                        "unspecified_site": docs["specimen_site"].eq("unspecified").astype(float).to_numpy(),
                    }
                    for name, values in covariates.items():
                        if np.ptp(values) > 0:
                            x[name] = values
                if np.linalg.matrix_rank(x.to_numpy()) < x.shape[1]:
                    row["status"] = "skipped: collinear design"
                else:
                    try:
                        with warnings.catch_warnings(record=True) as caught:
                            warnings.simplefilter("always")
                            fit = sm.GEE(y, x, groups=docs["DFCI_MRN"].to_numpy(dtype=np.int64),
                                         family=sm.families.Binomial(), cov_struct=sm.cov_struct.Independence()
                                         ).fit(maxiter=100, cov_type="robust")
                        beta, se = float(fit.params["risk_z"]), float(fit.bse["risk_z"])
                        if caught or not fit.converged or not np.isfinite([beta, se]).all() or se <= 0:
                            row["status"] = "skipped: fit warning, nonconvergence, or invalid covariance"
                        elif max(abs(beta - 1.96 * se), abs(beta + 1.96 * se)) > 700:
                            row["status"] = "skipped: unstable effect estimate"
                        else:
                            row.update(beta_per_sd=beta, odds_ratio_per_sd=np.exp(beta),
                                       ci95_low=np.exp(beta - 1.96 * se), ci95_high=np.exp(beta + 1.96 * se),
                                       p_value=float(fit.pvalues["risk_z"]))
                            if model == "gee":
                                grid = np.linspace(patients["risk_score"].min(), patients["risk_score"].max(), 100)
                                design = np.column_stack([np.ones(len(grid)), (grid - mean) / sd])
                                eta = design @ fit.params.to_numpy()
                                eta_se = np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", design, fit.cov_params(), design), 0))
                                curves.append(pd.DataFrame(dict(landmark=landmark, topic=topic,
                                    top_words=top_words.get(topic, ""), risk_score=grid, probability=expit(eta),
                                    ci95_low=expit(eta - 1.96 * eta_se), ci95_high=expit(eta + 1.96 * eta_se))))
                    except (ValueError, np.linalg.LinAlgError, FloatingPointError, OverflowError) as exc:
                        row["status"] = f"skipped: model fit failed ({type(exc).__name__})"
            results.append(row)
    effects = pd.DataFrame(results).reindex(columns=CONTINUOUS_COLS)
    curves = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame(columns=CURVE_COLS)
    return effects, curves


continuous_frames, curve_frames, continuous_coverage = [], [], []
if topic_docs is None:
    print("Continuous topic analysis skipped: BERTopic assignments unavailable.")
else:
    words = topic_table.set_index("topic")["top_words"].to_dict()
    for lm in LANDMARKS:
        g = by_lm[lm]
        risk = pd.Series(g["risk_score"].to_numpy(dtype=float),
                         index=pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64")))
        # Same report membership as the legacy analysis; all patients, regardless of decile label.
        lm_keys = rpt_key(reports_by_lm[lm]["RPT_ID"]).dropna().unique()
        docs = topic_docs[topic_docs["rpt_key"].isin(lm_keys)].copy()
        finite_ids = risk.index[np.isfinite(risk)]
        docs = docs[docs["DFCI_MRN"].isin(finite_ids)]
        continuous_coverage.append(dict(landmark=lm, patients_scored=len(finite_ids),
            patients_with_report=reports_by_lm[lm]["DFCI_MRN"].nunique(),
            patients_with_embedded_report=docs["DFCI_MRN"].nunique(), embedded_reports=len(docs),
            outlier_reports=int(docs["topic"].eq(-1).sum())))
        effects, curves = continuous_topic_association(docs, risk, lm, words)
        continuous_frames.append(effects)
        curve_frames.append(curves)
    continuous = pd.concat(continuous_frames, ignore_index=True).reindex(columns=CONTINUOUS_COLS)
    # One BH family per method over ALL tested topics and landmarks; failed/untestable rows remain NaN.
    for model, indices in continuous.groupby("model").groups.items():
        continuous.loc[indices, "q_value"] = bh_qvalues(continuous.loc[indices, "p_value"])
    continuous_curves = pd.concat(curve_frames, ignore_index=True).reindex(columns=CURVE_COLS)
    coverage = pd.DataFrame(continuous_coverage)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    continuous.to_csv(OUTPUT_DIR / "pathology_topic_continuous_association.csv", index=False)
    continuous_curves.to_csv(OUTPUT_DIR / "pathology_topic_risk_curves.csv", index=False)
    coverage.to_csv(OUTPUT_DIR / "pathology_topic_continuous_coverage.csv", index=False)
    display(coverage)
    for model in ("gee", "gee_adjusted", "patient_fraction_spearman"):
        sub = continuous[continuous["model"].eq(model)]
        print(f"\n{model}: {sub['status'].eq('ok').sum()} estimated; "
              f"{sub['q_value'].lt(TERM_FDR).sum()} at q < {TERM_FDR} across topics and landmarks")
        display(sub.sort_values("p_value").head(TERM_TOP_N).round(4).reset_index(drop=True))
    if not continuous_curves.empty:
        import matplotlib.pyplot as plt
        # Display the strongest unadjusted associations, including nonsignificant ones (q shown in each title).
        for _, row in continuous[continuous["model"].eq("gee") & continuous["status"].eq("ok")].nsmallest(6, "p_value").iterrows():
            curve = continuous_curves[continuous_curves["landmark"].eq(row["landmark"]) & continuous_curves["topic"].eq(row["topic"])]
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.plot(curve["risk_score"], curve["probability"])
            ax.fill_between(curve["risk_score"], curve["ci95_low"], curve["ci95_high"], alpha=0.2)
            ax.set(xlabel="Patient risk score (continuous)", ylabel="Fitted probability of topic in a note", ylim=(0, 1),
                   title=f"Landmark {int(row['landmark'])} · topic {int(row['topic'])} · q={row['q_value']:.3g}")
            fig.tight_layout()
            plt.show()
            plt.close(fig)


## Sensitivity and complementary analyses

Same groups, landmarks and reports (all dates) as the term enrichment. Each one addresses a specific way the term
tables can mislead.

1. **Report count and specimen type (`*_mh` columns).** Patients with more reports have more chances to mention any
   term, and metastatic-site biopsies read differently from prostate specimens. Each patient's stratum is their
   report-count band (`ADJUST_REPORT_BANDS`) crossed with whether any report is from a
   non-prostate site (from `PROC_DESC` and the report's first `SPECIMEN_HEADER_CHARS` characters). The term, topic
   and lexicon tables add a one-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio across strata.
   These are NaN when no stratum contains patients from both sides of the contrast.
2. **Curated lexicon (`pathology_lexicon_enrichment.csv`).** The neuroendocrine patterns are copied from the NEPC
   annotation pipeline's `ne_gate.py`, joined by individual NE and prostate IHC markers, histologic variants, invasion
   / extension terms and the highest Gleason sum (grade group mapped to a sum). Same Fisher and MH tests.
3. **Continuous risk (`pathology_score_association.csv`).** Deciles discard most of the score. For every term and
   lexicon feature, a two-sided Mann-Whitney test of the risk percentile among all scored patients with a
   report, feature present vs absent. `auc` = P(a patient with the feature ranks higher than one without). The van
   Elteren test (`*_adj`) stratifies it like item 1.
4. **Discrimination (`pathology_text_classifier.csv`).** Can the pathology text tell the groups apart at all? This is
   a cross-validated (`TEXT_FOLDS`) logistic regression on each patient's TF-IDF (L1) and, when the precomputed
   embeddings are available, on their mean report embedding (L2), compared with `TEXT_PERMUTATIONS` label
   permutations. Vocabulary selection (minimum distinct-patient frequency) and IDF are fitted within each training
   fold, including each permutation; bigrams never cross report boundaries. A fold with no retained terms uses an
   intercept-only prediction. TF-IDF `n_features` and top terms describe a separate fit on all patients in that contrast.

All outputs are aggregate. No MRN or report text is printed or written by these cells.


In [ ]:
# Curated lexicon, matched on lowercased report text. The NE patterns are copied verbatim (not imported) from
# LLM_clinical_annotations/tasks/nepc/ne_gate.py (NE_GATE_VERSION "ne-gate-v3", commit
# edb1713ef454f6ece313fa675b5f9438937f831e); update them by hand if that file changes.
NE_TERM = (r"\b(?:neuro[\s-]?endocrine|t-?nepc|nepc|small[\s-]?cell|scpc|scnc|"
           r"oat[\s-]?cell)\b")
SMALL_CELL_TERM = r"\b(?:small[\s-]?cell|scpc|scnc|oat[\s-]?cell)\b"
NE_DIAGNOSIS = (r"\b(?:t-?nepc|nepc|scpc|scnc)\b|"
                r"\b(?:neuro[\s-]?endocrine|small[\s-]?cell|oat[\s-]?cell)"
                r"(?:\s+[\w/()-]+){0,4}?\s+"
                r"(?:carcinoma|cancer|tumou?r|neoplasm|malignancy|component|histology|"
                r"variant|subtype|type|disease|prostate)\b|"
                r"\b(?:carcinoma|cancer)\s+(?:with|of)\s+(?:\w+\s+){0,2}?"
                r"(?:neuro[\s-]?endocrine|small[\s-]?cell)\b")
NE_MARKER = (r"\b(?:synaptophysin|chromogranin(?:\s+a)?|cd\s?56|insm-?1|"
             r"neuron[\s-]+specific\s+enolase|nse)\b")
NON_PROSTATE_NE = (r"\b(?:sclc|small[\s-]?cell\s+lung|merkel|carcinoid|"
                   r"pulmonary\s+(?:origin|primary|small)|lung\s+(?:origin|primary)|"
                   r"urothelial|bladder\s+(?:primary|origin|cancer|carcinoma|tumou?r)|"
                   r"pancreatic\s+(?:net|neuroendocrine)|pnet|gep[\s-]?nets?|"
                   r"well[\s-]differentiated\s+neuroendocrine\s+tumou?r)\b")
NON_SMALL_CELL = r"\bnon[\s-]*$"
# (end of the ne_gate.py copy)

LEXICON = {
    # neuroendocrine (ne_gate.py), then each NE marker from NE_MARKER on its own
    "NE term (any)": NE_TERM,
    "NE diagnosis phrase": NE_DIAGNOSIS,
    "small cell": SMALL_CELL_TERM,
    "NE marker (any)": NE_MARKER,
    "non-prostate NE primary": NON_PROSTATE_NE,
    "synaptophysin": r"\bsynaptophysin\b",
    "chromogranin": r"\bchromogranin\b",
    "CD56": r"\bcd\s?56\b",
    "INSM1": r"\binsm-?1\b",
    "NSE": r"\b(?:neuron[\s-]+specific\s+enolase|nse)\b",
    # other IHC
    "Ki-67 / MIB-1": r"\b(?:ki-?67|mib-?1)\b",
    "AR": r"\b(?:androgen\s+receptor|ar)\b",
    "NKX3.1": r"\bnkx\s?3[.\s-]?1\b",
    "PSMA": r"\b(?:psma|prostate[\s-]specific\s+membrane\s+antigen)\b",
    "ERG": r"\berg\b",
    "PTEN": r"\bpten\b",
    # histology and staging
    "ductal": r"\bductal\b",
    "intraductal": r"\bintraductal\b|\bidc-?p\b",
    "cribriform": r"\bcribriform\b",
    "sarcomatoid": r"\bsarcomatoid\b",
    "treatment effect": r"\b(?:treatment|therapy|hormon\w*|androgen[\s-]deprivation)[\s-]+(?:related\s+)?effects?\b",
    "perineural invasion": r"\bperineural\s+(?:invasion|infiltration)\b|\bpni\b",
    "lymphovascular invasion": (r"\b(?:lympho-?vascular|angiolymphatic|lymphatic|vascular)\s+(?:space\s+)?invasion\b"
                                r"|\blvi\b"),
    "extraprostatic extension": r"\bextra-?prostatic\s+extension\b|\bepe\b",
    "seminal vesicle invasion": r"\bseminal\s+vesicles?\s+(?:invasion|involvement|involved)\b|\bsvi\b",
}
LEXICON_RE = {name: re.compile(rx) for name, rx in LEXICON.items()}
NON_PREFIX_RE = re.compile(NON_SMALL_CELL)
# Gleason: the highest primary + secondary sum in any report (grade group mapped to a sum).
GLEASON_RE = re.compile(r"\bgleason\b[^.;\n]{0,40}?\b([1-5])\s*\+\s*([1-5])\b")
GRADE_GROUP_RE = re.compile(r"\b(?:grade\s+group|isup\s+grade(?:\s+group)?|gg)\s*:?\s*([1-5])\b")
GRADE_GROUP_SUM = {1: 6, 2: 7, 3: 7, 4: 8, 5: 9}
GLEASON_BINS = (("Gleason max 2-6", 2, 6), ("Gleason max 7", 7, 7), ("Gleason max 8-10", 8, 10))

MENTION_FEATURES = list(LEXICON)
LEXICON_FEATURES = MENTION_FEATURES + [label for label, _, _ in GLEASON_BINS]
LEXICON_COLS = ["landmark", "group", "control", "feature", "n_group", "n_control", "group_with_feature",
                "control_with_feature", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS]


def lexicon_hits(text):
    """One report's lexicon features: {feature: True} for each mention, plus gleason_max.

    A match right after "non-" ("non-small cell") is not a mention.
    """
    text = text.lower()
    hits = {}
    for name, rx in LEXICON_RE.items():
        if any(not NON_PREFIX_RE.search(text[max(0, m.start() - 8):m.start()]) for m in rx.finditer(text)):
            hits[name] = True
    sums = ([int(a) + int(b) for a, b in GLEASON_RE.findall(text)]
            + [GRADE_GROUP_SUM[int(g)] for g in GRADE_GROUP_RE.findall(text)])
    if sums:
        hits["gleason_max"] = max(sums)
    return hits


# Report-level hits, once for every report at any landmark.
report_rows = np.unique(np.concatenate([rep["note_row"].to_numpy() for rep in reports_by_lm.values()]
                                       + [np.array([], dtype=int)])).astype(int)
report_lex = pd.DataFrame([lexicon_hits(t) for t in all_notes["RPT_TEXT"].to_numpy()[report_rows]],
                          index=report_rows).reindex(columns=MENTION_FEATURES + ["gleason_max"])
report_mentions = report_lex[MENTION_FEATURES].notna().to_numpy()
report_gleason = pd.to_numeric(report_lex["gleason_max"], errors="coerce").to_numpy(dtype=float)
lexicon_matrices = {}   # landmark -> (patient ids, presence); reused by the score association cell
lexicon_frames = []
for lm in LANDMARKS:
    if lm not in reports_by_lm or reports_by_lm[lm].empty:
        continue
    rep = reports_by_lm[lm]
    ids = pd.Index(sorted(rep["DFCI_MRN"].unique()), dtype="Int64")
    rows = ids.get_indexer(rep["DFCI_MRN"])
    pos = np.searchsorted(report_rows, rep["note_row"].to_numpy())
    mentions = np.zeros((len(ids), len(MENTION_FEATURES)), dtype=bool)
    np.logical_or.at(mentions, rows, report_mentions[pos])
    gleason = np.full(len(ids), np.nan)
    np.fmax.at(gleason, rows, report_gleason[pos])
    bins = np.column_stack([(gleason >= lo) & (gleason <= hi) for _, lo, hi in GLEASON_BINS])
    present = sparse.csr_matrix(np.hstack([mentions, bins]).astype(np.int64))
    lexicon_matrices[lm] = (ids, present)
    # Only features some patient has at this landmark are tested, so absent ones do not dilute the BH correction.
    seen = np.flatnonzero(colsum(present) > 0)
    lexicon_frames.append(fisher_enrichment(present[:, seen], groups_by_lm[lm].reindex(ids).to_numpy(),
                                            np.array(LEXICON_FEATURES, dtype=object)[seen], lm,
                                            strata_by_lm[lm].reindex(ids).to_numpy()))

# Rows stay in landmark, CONTRASTS, then LEXICON_FEATURES order.
lexicon_enrich = (pd.concat(lexicon_frames, ignore_index=True) if lexicon_frames else pd.DataFrame()
                  ).reindex(columns=LEXICON_COLS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
lexicon_path = OUTPUT_DIR / "pathology_lexicon_enrichment.csv"
lexicon_enrich.to_csv(lexicon_path, index=False)
print(f"wrote {lexicon_path} ({len(lexicon_enrich):,} feature x contrast rows)")

# One wide table per landmark: % of patients with each feature by group, then q values for every contrast
# (q = Fisher, q_mh = stratified Mantel-Haenszel). Features no patient has at that landmark are not tested.
for lm in LANDMARKS:
    sub = lexicon_enrich[lexicon_enrich["landmark"] == lm]
    if sub.empty:
        print(f"\nLandmark {lm} · lexicon: no patients with pathology reports")
        continue
    wide = pd.DataFrame(index=pd.Index(LEXICON_FEATURES, name="feature"))
    for name, ref in CONTRASTS:
        c = sub[(sub["group"] == name) & (sub["control"] == ref)].set_index("feature")
        if c.empty:
            continue
        wide[f"% {name}"], wide[f"% {ref}"] = c["pct_group"] * 100, c["pct_control"] * 100
        wide[f"q {name}>{ref}"], wide[f"q_mh {name}>{ref}"] = c["q_value"], c["q_value_mh"]
    pct_cols = [f"% {g}" for g in (TERM_GROUPS[0], CONTROL_GROUP, TERM_GROUPS[1]) if f"% {g}" in wide]
    q_cols = [col for prefix in ("q ", "q_mh ") for col in wide.columns if col.startswith(prefix)]
    wide = wide[pct_cols + q_cols]
    print(f"\nLandmark {lm} · curated lexicon (% of patients with a mention in any report)")
    display(wide.dropna(how="all").round(3))


In [ ]:
from scipy.stats import rankdata

ASSOC_COLS = ["landmark", "feature_set", "feature", "n_patients", "n_with_feature", "pct_with_feature",
              "mean_percentile_with", "mean_percentile_without", "auc", "p_value", "q_value",
              "auc_adj", "p_value_adj", "q_value_adj"]


def _mann_whitney_parts(present, score):
    """Per feature column: n with it, U of the with-feature patients, and the tie-corrected null variance of U."""
    n = present.shape[0]
    ranks = rankdata(score)
    n1 = colsum(present)
    u = np.asarray(present.T @ ranks).ravel() - n1 * (n1 + 1) / 2
    _, t = np.unique(score, return_counts=True)
    ties = float((t.astype(float) ** 3 - t).sum()) / (n * (n - 1)) if n > 1 else 0.0
    var = n1 * (n - n1) / 12 * ((n + 1) - ties)
    return n1, u, var


def score_association(present, features, score, strata, landmark, feature_set):
    """Two-sided Mann-Whitney of the risk percentile, feature present vs absent, among all scored patients with
    pathology reports; and the van Elteren test stratified like the Mantel-Haenszel columns.

    auc = P(percentile of a patient with the feature > one without); auc_adj pools the same comparison within strata.
    """
    present = present.tocsr().astype(float)
    n = present.shape[0]
    n1, u, var = _mann_whitney_parts(present, score)
    n0 = n - n1
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.maximum(np.abs(u - n1 * n0 / 2) - 0.5, 0) / np.sqrt(var)
        p = np.where(var > 0, 2 * norm.sf(z), np.nan)
        sum_with = np.asarray(present.T @ score).ravel()
        res = pd.DataFrame({
            "landmark": landmark, "feature_set": feature_set, "feature": features, "n_patients": n,
            "n_with_feature": n1.astype(int), "pct_with_feature": n1 / n,
            "mean_percentile_with": sum_with / n1, "mean_percentile_without": (score.sum() - sum_with) / n0,
            "auc": u / (n1 * n0), "p_value": p})
    # van Elteren: within-stratum U, strata weighted 1 / (N_k + 1).
    num, den, u_pool, pairs = (np.zeros(len(features)) for _ in range(4))
    for k in pd.unique(strata):
        rows = np.flatnonzero(strata == k)
        if rows.size < 2:
            continue
        n1k, uk, vk = _mann_whitney_parts(present[rows], score[rows])
        w = 1 / (rows.size + 1)
        num += w * (uk - n1k * (rows.size - n1k) / 2)
        den += w ** 2 * vk
        u_pool += uk
        pairs += n1k * (rows.size - n1k)
    with np.errstate(divide="ignore", invalid="ignore"):
        res["auc_adj"] = np.where(pairs > 0, u_pool / pairs, np.nan)
        res["p_value_adj"] = np.where(den > 0, 2 * norm.sf(np.abs(num) / np.sqrt(den)), np.nan)
    res = res[(res["n_with_feature"] > 0) & (res["n_with_feature"] < n)].reset_index(drop=True)
    res["q_value"], res["q_value_adj"] = bh_qvalues(res["p_value"]), bh_qvalues(res["p_value_adj"])
    return res[ASSOC_COLS]


assoc_frames = []
for lm in LANDMARKS:
    if lm not in by_lm:
        continue
    g = by_lm[lm]
    pct = pd.Series(g["risk_percentile"].to_numpy(dtype=float),
                    index=pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64")))
    sets = []
    if lm in term_matrices and len(term_matrices[lm][1]):
        ids, terms, present, _ = term_matrices[lm]
        sets.append(("term", ids, present, terms))
    if lm in lexicon_matrices:
        ids, present = lexicon_matrices[lm]
        sets.append(("lexicon", ids, present, np.array(LEXICON_FEATURES, dtype=object)))
    for feature_set, ids, present, names in sets:
        if len(ids) < 2:
            continue
        assoc_frames.append(score_association(present, names, pct.reindex(ids).to_numpy(dtype=float),
                                              strata_by_lm[lm].reindex(ids).to_numpy(), lm, feature_set))

assoc = (pd.concat(assoc_frames, ignore_index=True) if assoc_frames else pd.DataFrame()).reindex(columns=ASSOC_COLS)
assoc_path = OUTPUT_DIR / "pathology_score_association.csv"
assoc.to_csv(assoc_path, index=False)
print(f"wrote {assoc_path} ({len(assoc):,} feature x landmark rows)")

assoc_show = ["feature", "n_with_feature", "pct_with_feature", "mean_percentile_with", "mean_percentile_without",
              "auc", "q_value", "auc_adj", "q_value_adj"]
for lm in LANDMARKS:
    for feature_set in ("term", "lexicon"):
        sub = assoc[(assoc["landmark"] == lm) & (assoc["feature_set"] == feature_set)]
        if sub.empty:
            print(f"\nLandmark {lm} · {feature_set}s vs risk percentile: none tested")
            continue
        for direction, side in (("higher", sub["auc"] > 0.5), ("lower", sub["auc"] < 0.5)):
            hits = sub[side].sort_values("p_value")
            print(f"\nLandmark {lm} · {feature_set}s with {direction} risk percentiles "
                  f"(q < {TERM_FDR}: {int((hits['q_value'] < TERM_FDR).sum()):,}; top {TERM_TOP_N} by p)")
            display(hits[assoc_show].head(TERM_TOP_N).round(3).reset_index(drop=True))


In [ ]:
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CLASSIFIER_PAIRS = (("top", CONTROL_GROUP), ("bottom", CONTROL_GROUP), ("top", "bottom"))   # (positive, negative)
CLASSIFIER_COLS = ["landmark", "positive_group", "negative_group", "features", "n_positive", "n_negative",
                   "n_features", "cv_auc", "null_auc_mean", "null_auc_95", "p_permutation",
                   "top_terms_positive", "top_terms_negative", "note"]
CLASSIFIER_TOP_TERMS = 10


class PatientTfidf(TransformerMixin, BaseEstimator):
    """Fit vocabulary and IDF on training patients, keeping each patient's report boundaries intact."""

    def __init__(self, min_patients=5, ngram_range=(1, 2), token_pattern=TOKEN_PATTERN):
        self.min_patients = min_patients
        self.ngram_range = ngram_range
        self.token_pattern = token_pattern

    def _terms(self, reports):
        for report in reports:
            yield from self.analyzer_(report)

    def fit(self, X, y=None):
        self.analyzer_ = CountVectorizer(lowercase=True, stop_words="english", token_pattern=self.token_pattern,
                                         ngram_range=self.ngram_range).build_analyzer()
        # Each input document is one patient's list of reports: min_df counts patients, not reports.
        self.vectorizer_ = CountVectorizer(analyzer=self._terms, min_df=self.min_patients)
        self.empty_ = len(X) < self.min_patients
        if not self.empty_:
            try:
                counts = self.vectorizer_.fit_transform(X)
            except ValueError as exc:
                if not any(message in str(exc) for message in ("empty vocabulary", "After pruning, no terms remain")):
                    raise
                self.empty_ = True
        if self.empty_:
            # A constant zero column lets logistic regression fit an intercept when a fold has no usable terms.
            self.names_ = np.array([""], dtype=object)
        else:
            self.names_ = self.vectorizer_.get_feature_names_out()
            self.tfidf_ = TfidfTransformer(sublinear_tf=True).fit(counts)
        return self

    def transform(self, X):
        if self.empty_:
            return sparse.csr_matrix((len(X), 1), dtype=float)
        return self.tfidf_.transform(self.vectorizer_.transform(X))

    def get_feature_names_out(self, input_features=None):
        return self.names_.copy()


def text_model(features):
    """L1 logistic on TF-IDF (sparse, so the kept terms are readable); L2 logistic on standardized embeddings."""
    if features == "embedding":
        return make_pipeline(StandardScaler(), LogisticRegression(C=TEXT_C, class_weight="balanced", max_iter=2000))
    kw = dict(solver="liblinear", C=TEXT_C, class_weight="balanced", max_iter=1000)
    # scikit-learn 1.8 deprecated penalty= in favour of l1_ratio=; older releases (the cluster) ignore l1_ratio here.
    if tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8):
        classifier = LogisticRegression(l1_ratio=1.0, **kw)
    else:
        classifier = LogisticRegression(penalty="l1", **kw)
    return make_pipeline(PatientTfidf(min_patients=TERM_MIN_PATIENTS, ngram_range=TERM_NGRAMS), classifier)


def patient_embeddings(lm):
    """Mean precomputed embedding over each patient's embedded reports at landmark lm."""
    if report_embeddings is None:
        return None, None
    keys = rpt_key(reports_by_lm[lm]["RPT_ID"]).dropna().unique()
    pos = embed_pos.reindex(keys).dropna().astype(int).to_numpy()
    if not pos.size:
        return None, None
    mrn = embed_docs["DFCI_MRN"].to_numpy()[pos]
    ids = pd.Index(sorted(set(mrn)), dtype="Int64")
    rows = ids.get_indexer(mrn)
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))), shape=(len(ids), len(rows)))
    to_patient = sparse.diags(1 / np.asarray(to_patient.sum(axis=1)).ravel()) @ to_patient
    return ids, np.asarray(to_patient @ report_embeddings[pos])


def classify(X, y, features):
    model = text_model(features)
    cv = StratifiedKFold(TEXT_FOLDS, shuffle=True, random_state=TEXT_SEED)
    auc, null, p = permutation_test_score(model, X, y, cv=cv, n_permutations=TEXT_PERMUTATIONS, scoring="roc_auc",
                                          n_jobs=N_JOBS, random_state=TEXT_SEED)
    row = {"cv_auc": auc, "null_auc_mean": float(np.mean(null)), "null_auc_95": float(np.quantile(null, 0.95)),
           "p_permutation": p}
    if features == "tfidf":   # descriptive fit on all contrast patients, separate from the CV fits
        model.fit(X, y)
        names = model.steps[0][1].get_feature_names_out()
        row["n_features"] = int(np.count_nonzero(names != ""))
        coef = model.steps[-1][1].coef_.ravel()
        order = np.argsort(coef)
        row["top_terms_positive"] = ", ".join(names[i] for i in order[::-1][:CLASSIFIER_TOP_TERMS] if names[i] and coef[i] > 0)
        row["top_terms_negative"] = ", ".join(names[i] for i in order[:CLASSIFIER_TOP_TERMS] if names[i] and coef[i] < 0)
    return row


embed_pos = (pd.Series(np.arange(len(embed_docs)), index=embed_docs["rpt_key"].to_numpy())
             if embed_docs is not None else None)
if report_embeddings is None:
    print(f"Embedding classifier skipped ({embed_skip}); TF-IDF only.")

clf_rows = []
for lm in LANDMARKS:
    if lm not in reports_by_lm:
        continue
    grp = groups_by_lm[lm]
    feature_sets = {}
    reports = reports_by_lm[lm]
    if not reports.empty:
        pooled = reports.groupby("DFCI_MRN", sort=True)["RPT_TEXT"].agg(list)
        # A one-dimensional object array keeps whole patients together when sklearn indexes CV folds.
        documents = np.empty(len(pooled), dtype=object)
        documents[:] = pooled.tolist()
        feature_sets["tfidf"] = (pooled.index, documents)
    e_ids, e_x = patient_embeddings(lm)
    if e_ids is not None:
        feature_sets["embedding"] = (e_ids, e_x)
    for pos_name, neg_name in CLASSIFIER_PAIRS:
        for features, (f_ids, X) in feature_sets.items():
            label = grp.reindex(f_ids).to_numpy()
            keep = np.flatnonzero((label == pos_name) | (label == neg_name))
            y = (label[keep] == pos_name).astype(int)
            row = {"landmark": lm, "positive_group": pos_name, "negative_group": neg_name, "features": features,
                   "n_positive": int(y.sum()), "n_negative": int(len(y) - y.sum()), "n_features": X.shape[1] if features == "embedding" else np.nan}
            if min(row["n_positive"], row["n_negative"]) < TEXT_FOLDS:
                row["note"] = f"skipped: fewer than TEXT_FOLDS = {TEXT_FOLDS} patients in a group"
            else:
                row.update(classify(X[keep], y, features))
            clf_rows.append(row)

classifier = pd.DataFrame(clf_rows).reindex(columns=CLASSIFIER_COLS)
classifier_path = OUTPUT_DIR / "pathology_text_classifier.csv"
classifier.to_csv(classifier_path, index=False)
print(f"wrote {classifier_path} ({len(classifier):,} rows)")
print(f"\nCross-validated AUC ({TEXT_FOLDS}-fold) vs {TEXT_PERMUTATIONS} label permutations; "
      f"p resolution 1/{TEXT_PERMUTATIONS + 1}")
display(classifier.drop(columns=["top_terms_positive", "top_terms_negative"]).round(3))
for row in classifier.dropna(subset=["cv_auc"]).itertuples(index=False):
    if row.features == "tfidf":
        print(f"\nLandmark {row.landmark} · {row.positive_group} vs {row.negative_group} (TF-IDF, AUC {row.cv_auc:.2f})"
              f"\n  toward {row.positive_group}: {row.top_terms_positive or '(none)'}"
              f"\n  toward {row.negative_group}: {row.top_terms_negative or '(none)'}")
